# B-Trees: Forensic Exploration with SQLite

**Data Mastery Lab** — Salesforce Data Cloud

---

## What we'll explore

1. **Visual B+Tree construction** — watch inserts cause splits and tree growth
2. **Write amplification** — every INSERT is O(log N) comparisons + potential cascading splits
3. **SQLite forensics** — inspect real B-tree pages inside a `.db` file
4. **WAL (Write-Ahead Log)** — how SQLite ensures durability without rewriting B-tree pages on every commit

### The core insight

> A B-tree is an **optimization for reads at the cost of writes**.
> Every write must maintain sorted order across a balanced tree — that's the tax you pay.

| Operation | B-Tree Cost | Why |
|-----------|-------------|-----|
| Point lookup | O(log_B N) | Tree traversal, one page per level |
| Insert | O(log_B N) + splits | Find leaf + insert + possible cascade |
| Range scan | O(log_B N + K) | Find start, then scan K leaf pages |
| Full scan | O(N) | Touch every leaf page |

Where B = branching factor (keys per node), N = total keys.

---
## Part 1: Visualizing B+Tree Construction

We'll use a pure-Python B+Tree with order=4 (max 4 children per internal node, max 3 keys per node).
This mirrors how SQLite's B-tree pages work — each page holds up to N keys and splits when full.

In [ ]:
import sys, os, shutil
sys.path.insert(0, '.')
from btree_viz import BPlusTree, render_tree, ascii_tree
from IPython.display import Image, display, HTML
import math

# All generated files (images, DBs, dot files) go into _output/
OUTPUT_DIR = "_output"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)
print(f"Output directory: {OUTPUT_DIR}/")

In [3]:
# Order-4 B+Tree: each node holds at most 3 keys.
# When a 4th key arrives → SPLIT.

tree = BPlusTree(order=4)
insert_sequence = [10, 20, 5, 15, 25, 30, 35, 40, 3, 7, 12, 18, 22, 28, 33, 38]

print("Insert-by-insert trace:")
print(f"{'Key':>5} {'Comparisons':>12} {'Splits':>7} {'Height':>7} {'Total Keys':>11}")
print("-" * 50)

for key in insert_sequence:
    event = tree.insert(key)
    split_marker = " ◄ SPLIT" if event['splits'] > 0 else ""
    print(f"{key:5d} {event['comparisons']:12d} {event['splits']:7d} "
          f"{event['tree_height']:7d} {event['total_keys']:11d}{split_marker}")

Insert-by-insert trace:
  Key  Comparisons  Splits  Height  Total Keys
--------------------------------------------------
   10            0       0       1           1
   20            1       0       1           2
    5            1       0       1           3
   15            3       1       2           5 ◄ SPLIT
   25            3       0       2           6
   30            4       1       2           8 ◄ SPLIT
   35            4       0       2           9
   40            5       1       2          11 ◄ SPLIT
    3            2       0       2          12
    7            4       2       3          14 ◄ SPLIT
   12            5       0       3          15
   18            5       0       3          16
   22            6       1       3          18 ◄ SPLIT
   28            4       0       3          19
   33            5       1       3          21 ◄ SPLIT
   38            5       0       3          22


In [ ]:
# Visualize the final tree
path = render_tree(tree, "btree_final", output_dir=OUTPUT_DIR)
if path.endswith(".png"):
    display(Image(filename=path))
else:
    print("Graphviz not found — showing ASCII + DOT source")
    print(ascii_tree(tree))
    print(f"\nDOT file saved to: {path}")
    print("Paste contents into https://dreampuf.github.io/GraphvizOnline/")

### Step-by-step split visualization

Let's rebuild the tree and capture snapshots at every split.

In [ ]:
# Rebuild with snapshots at every insert
tree2 = BPlusTree(order=4)
snapshots = []

for i, key in enumerate(insert_sequence):
    event = tree2.insert(key)
    if event['splits'] > 0 or i < 4:  # always show first 4 + every split
        fname = f"step_{i:02d}_insert_{key}"
        path = render_tree(tree2, fname, output_dir=OUTPUT_DIR, highlight_key=key)
        snapshots.append((key, event, path))

for key, event, path in snapshots:
    print(f"\n── After inserting {key} (splits={event['splits']}, height={event['tree_height']}) ──")
    if path.endswith(".png"):
        display(Image(filename=path))
    else:
        print(ascii_tree(tree2))

### Key takeaway: Write amplification

Every INSERT into a B-tree is NOT just "append a row". It's:

1. **Traverse** the tree to find the right leaf — O(log_B N) comparisons
2. **Insert** the key in sorted position within the leaf
3. If the leaf is full → **SPLIT**: create a new node, redistribute keys, promote a key to parent
4. If the parent overflows → **CASCADE**: split the parent too (can propagate to root)
5. If root splits → **tree grows taller** (the ONLY way a B-tree gets taller)

This is the fundamental **read-write tradeoff** of B-trees:
- Reads are fast (balanced tree, O(log N))
- Writes pay the cost of maintaining that balance

---
## Part 2: Measuring the Cost — Insert Overhead at Scale

In [6]:
import random

# Insert 10,000 keys and track cumulative splits & comparisons
tree_large = BPlusTree(order=100)  # SQLite pages typically hold ~100 keys
random.seed(42)
keys_to_insert = random.sample(range(1, 100_001), 10_000)

cum_splits = []
cum_comparisons = []
heights = []
total_comparisons = 0

for k in keys_to_insert:
    event = tree_large.insert(k)
    total_comparisons += event['comparisons']
    cum_splits.append(tree_large.split_count)
    cum_comparisons.append(total_comparisons)
    heights.append(event['tree_height'])

print(f"After 10,000 inserts (order=100):")
print(f"  Tree height: {tree_large.height()}")
print(f"  Total splits: {tree_large.split_count}")
print(f"  Total comparisons: {total_comparisons:,}")
print(f"  Avg comparisons/insert: {total_comparisons / 10_000:.1f}")
print(f"  Theoretical O(log_100(10000)): {math.log(10_000, 100):.2f}")

After 10,000 inserts (order=100):
  Tree height: 3
  Total splits: 142
  Total comparisons: 638,603
  Avg comparisons/insert: 63.9
  Theoretical O(log_100(10000)): 2.00


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cumulative splits
axes[0].plot(range(1, 10_001), cum_splits, color='red', linewidth=0.8)
axes[0].set_title('Cumulative Splits Over Inserts')
axes[0].set_xlabel('Insert #')
axes[0].set_ylabel('Total Splits')
axes[0].grid(True, alpha=0.3)

# Comparisons per insert (rolling average)
window = 100
per_insert_comp = [cum_comparisons[0]] + \
    [cum_comparisons[i] - cum_comparisons[i-1] for i in range(1, len(cum_comparisons))]
rolling = [sum(per_insert_comp[max(0,i-window):i+1]) / min(i+1, window) 
           for i in range(len(per_insert_comp))]
axes[1].plot(range(1, 10_001), rolling, color='blue', linewidth=0.8)
axes[1].axhline(y=math.log(10_000, 100), color='red', linestyle='--', 
                label=f'O(log₁₀₀(N)) = {math.log(10_000, 100):.1f}')
axes[1].set_title('Avg Comparisons per Insert (rolling)')
axes[1].set_xlabel('Insert #')
axes[1].set_ylabel('Comparisons')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Tree height
axes[2].plot(range(1, 10_001), heights, color='green', linewidth=1.5)
axes[2].set_title('Tree Height Over Inserts')
axes[2].set_xlabel('Insert #')
axes[2].set_ylabel('Height')
axes[2].set_ylim(0, max(heights) + 1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('B+Tree Insert Overhead (order=100, 10K random inserts)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/insert_overhead.png', dpi=150)
plt.show()
print("\n↑ Notice: tree height increases in discrete JUMPS (root splits).")
print("  Each height increase means EVERY future operation costs 1 more page read.")

### Sequential vs Random inserts — the hidden cost

In [8]:
# Compare sequential vs random inserts
tree_seq = BPlusTree(order=100)
tree_rnd = BPlusTree(order=100)

n = 10_000
random.seed(42)
random_keys = random.sample(range(1, n * 10), n)
sequential_keys = list(range(1, n + 1))

seq_comparisons = 0
rnd_comparisons = 0

for k in sequential_keys:
    seq_comparisons += tree_seq.insert(k)['comparisons']

for k in random_keys:
    rnd_comparisons += tree_rnd.insert(k)['comparisons']

print("                  Sequential    Random")
print(f"Total comparisons: {seq_comparisons:>10,}  {rnd_comparisons:>10,}")
print(f"Avg per insert:    {seq_comparisons/n:>10.1f}  {rnd_comparisons/n:>10.1f}")
print(f"Total splits:      {tree_seq.split_count:>10}  {tree_rnd.split_count:>10}")
print(f"Final height:      {tree_seq.height():>10}  {tree_rnd.height():>10}")
print()
print("💡 Sequential inserts always hit the rightmost leaf → fewer comparisons")
print("   but cause WORST page utilization (~50%). Random fills pages more evenly (~69%).")
print("   This is why auto-increment PKs are fast to insert but waste space.")

                  Sequential    Random
Total comparisons:  1,360,050     638,603
Avg per insert:         136.0        63.9
Total splits:             201         142
Final height:               3           3

💡 Sequential inserts always hit the rightmost leaf → fewer comparisons
   but cause WORST page utilization (~50%). Random fills pages more evenly (~69%).
   This is why auto-increment PKs are fast to insert but waste space.


---
## Part 3: SQLite Forensics — Real B-Trees on Disk

Now let's move from theory to practice. SQLite stores **everything** in B-trees:
- **Table B-tree**: stores rows, keyed by rowid (B+tree — data in leaves)
- **Index B-tree**: stores index entries (B-tree — keys in all nodes)

Let's create a real database and inspect it.

In [ ]:
import sqlite3

DB_PATH = f"{OUTPUT_DIR}/lab_btree.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Create a table — this creates a Table B-tree
cur.execute('''
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT NOT NULL,
        phone TEXT NOT NULL,
        department TEXT,
        salary REAL
    )
''')

# Index on HIGH CARDINALITY columns — email and phone are unique per person.
# These are the columns where B-tree indexes shine: each lookup narrows to 1 row.
# (Indexing low-cardinality columns like department would be wasteful —
#  a column with only 20 distinct values doesn't benefit from a B-tree.)
cur.execute('CREATE UNIQUE INDEX idx_email ON employees(email)')
cur.execute('CREATE UNIQUE INDEX idx_phone ON employees(phone)')

conn.commit()
print("Created table + 2 indexes on email & phone (= 3 B-trees on disk)")
print("  → High cardinality: every email/phone is unique → B-tree lookups go straight to 1 row")
print("  → Low cardinality columns like 'department' (20 values) are NOT worth indexing")

In [ ]:
# Insert data in batches and watch the file grow
import time

departments = [f"Dept_{i}" for i in range(1, 21)]
batch_sizes = [100, 500, 1000, 5000, 10000]
total_inserted = 0

print(f"{'Rows':>8} {'DB Size':>12} {'Insert Time':>12} {'Pages':>8}")
print("-" * 45)

for batch in batch_sizes:
    start = time.time()
    rows = []
    for i in range(total_inserted + 1, total_inserted + batch + 1):
        dept = random.choice(departments)
        salary = random.uniform(50000, 200000)
        name = f"Employee_{i:06d}"
        email = f"emp{i:06d}@company.com"
        phone = f"+1-555-{i:07d}"
        rows.append((i, name, email, phone, dept, salary))
    cur.executemany("INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?)", rows)
    conn.commit()
    elapsed = time.time() - start
    total_inserted += batch
    
    db_size = os.path.getsize(DB_PATH)
    page_count = cur.execute("PRAGMA page_count").fetchone()[0]
    
    print(f"{total_inserted:8,} {db_size:10,} B {elapsed:10.3f}s {page_count:8,}")

print(f"\nPage size: {cur.execute('PRAGMA page_size').fetchone()[0]} bytes")
print(f"Each page = one B-tree node on disk")

In [11]:
# SQLite internal stats — which B-trees exist and how big are they?
print("=== B-Trees in this database ===")
for row in cur.execute("SELECT type, name, rootpage FROM sqlite_master").fetchall():
    print(f"  {row[0]:>5}  {row[1]:<20}  root_page={row[2]}")

print("\n=== B-Tree page stats (via sqlite_dbstat) ===")
# dbstat virtual table gives per-page stats
try:
    stats = cur.execute('''
        SELECT name, 
               COUNT(*) as pages,
               SUM(payload) as total_payload,
               SUM(unused) as total_unused,
               MAX(mx_payload) as max_payload
        FROM dbstat 
        GROUP BY name
    ''').fetchall()
    print(f"{'Name':<22} {'Pages':>6} {'Payload':>12} {'Unused':>12} {'Fill%':>6}")
    print("-" * 62)
    for name, pages, payload, unused, max_p in stats:
        page_size = 4096
        total_space = pages * page_size
        fill = (payload / total_space * 100) if total_space > 0 else 0
        print(f"{name:<22} {pages:>6} {payload:>10,} B {unused:>10,} B {fill:>5.1f}%")
except Exception as e:
    print(f"dbstat not available: {e}")
    print("(Compile SQLite with SQLITE_ENABLE_DBSTAT_VTAB for full stats)")

=== B-Trees in this database ===
  table  employees             root_page=2
  index  idx_dept              root_page=3
  index  idx_salary            root_page=4

=== B-Tree page stats (via sqlite_dbstat) ===
Name                    Pages      Payload       Unused  Fill%
--------------------------------------------------------------
employees                 140    480,550 B      7,571 B  83.8%
idx_dept                   41     98,622 B     19,026 B  58.7%
idx_salary                 73    215,672 B     32,664 B  72.1%
sqlite_schema               1        325 B      3,650 B   7.9%


### The INSERT transaction breakdown

Every single INSERT into `employees` actually touches **3 B-trees**:

1. **Table B-tree** (`employees`): insert the row data
2. **Index B-tree** (`idx_email`): insert `(email, rowid)` entry
3. **Index B-tree** (`idx_phone`): insert `(phone, rowid)` entry

Each B-tree insertion is O(log N). So one INSERT = **3 × O(log N)** B-tree operations.

More indexes = more write overhead. This is the fundamental **index tax**.

Note: we deliberately indexed **high-cardinality** columns (email, phone — unique per row).
Indexing a low-cardinality column like `department` (only 20 distinct values) would waste space
and add write overhead for minimal read benefit — the B-tree would still scan ~5% of all rows.

In [12]:
# Demonstrate the index tax: measure insert speed with 0, 1, 3, 5 indexes

def measure_insert_speed(n_indexes, n_rows=10000):
    db = sqlite3.connect(":memory:")
    c = db.cursor()
    c.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, a INT, b INT, c INT, d INT, e INT)")
    
    # Create requested number of indexes
    cols = ['a', 'b', 'c', 'd', 'e']
    for i in range(n_indexes):
        c.execute(f"CREATE INDEX idx_{cols[i]} ON t({cols[i]})")
    
    random.seed(42)
    rows = [(i, random.randint(1,100000), random.randint(1,100000),
             random.randint(1,100000), random.randint(1,100000),
             random.randint(1,100000)) for i in range(n_rows)]
    
    start = time.time()
    c.executemany("INSERT INTO t VALUES (?,?,?,?,?,?)", rows)
    db.commit()
    elapsed = time.time() - start
    
    db.close()
    return elapsed

print("Index Tax: INSERT speed vs number of indexes")
print(f"{'Indexes':>8} {'Time (s)':>10} {'Rows/sec':>12} {'Relative':>10}")
print("-" * 45)

baseline = None
for n_idx in [0, 1, 2, 3, 5]:
    elapsed = measure_insert_speed(n_idx)
    rps = 10000 / elapsed
    if baseline is None:
        baseline = elapsed
    relative = elapsed / baseline
    print(f"{n_idx:>8} {elapsed:>10.4f} {rps:>12,.0f} {relative:>9.2f}x")

print("\n↑ Each additional index adds another B-tree that must be maintained on every write.")

Index Tax: INSERT speed vs number of indexes
 Indexes   Time (s)     Rows/sec   Relative
---------------------------------------------
       0     0.0106      940,659      1.00x
       1     0.0091    1,103,416      0.85x
       2     0.0121      827,589      1.14x
       3     0.0137      732,220      1.28x
       5     0.0191      524,183      1.79x

↑ Each additional index adds another B-tree that must be maintained on every write.


---
## Part 4: WAL — Write-Ahead Logging

We've seen that every INSERT modifies B-tree pages. But rewriting pages directly on disk for every transaction would be:
1. **Slow** — random I/O to update pages scattered across the file
2. **Unsafe** — a crash mid-write could corrupt the B-tree

SQLite solves this with **WAL (Write-Ahead Log)**:
- Writes go to a **sequential append-only log file** first (fast!)
- The actual B-tree pages in the main `.db` file are updated later (checkpoint)
- On crash recovery: replay the WAL to reconstruct committed state

This is the same principle behind every serious database: PostgreSQL's WAL, MySQL's redo log, Oracle's redo log.

In [ ]:
# Switch to WAL mode and observe the files
print("Current journal mode:", cur.execute("PRAGMA journal_mode").fetchone()[0])

# Switch to WAL
cur.execute("PRAGMA journal_mode=WAL")
print("Switched to:", cur.execute("PRAGMA journal_mode").fetchone()[0])

# Show what files exist now
import glob
print("\nFiles on disk:")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/lab_btree*")):
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f):<30} {size:>10,} bytes")

print("\n→ The -wal file is the Write-Ahead Log")
print("→ The -shm file is shared memory for concurrent readers")

In [ ]:
# Watch the WAL grow with writes, while the main DB stays the same size

db_size_before = os.path.getsize(DB_PATH)
wal_path = DB_PATH + "-wal"

print("Inserting 5,000 more rows and watching WAL vs DB size...\n")
print(f"{'After':>8} {'DB File':>12} {'WAL File':>12} {'WAL Pages':>10}")
print("-" * 48)

for batch_num in range(5):
    rows = []
    base = total_inserted + batch_num * 1000 + 1
    for i in range(base, base + 1000):
        rows.append((i, f"Employee_{i:06d}", f"emp{i:06d}@company.com",
                      f"+1-555-{i:07d}", random.choice(departments),
                      random.uniform(50000, 200000)))
    cur.executemany("INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?)", rows)
    conn.commit()
    
    db_size = os.path.getsize(DB_PATH)
    wal_size = os.path.getsize(wal_path) if os.path.exists(wal_path) else 0
    wal_pages = wal_size // 4096  # approximate
    
    print(f"{(batch_num+1)*1000:>8} {db_size:>10,} B {wal_size:>10,} B {wal_pages:>10}")

print(f"\nDB file size change: {os.path.getsize(DB_PATH) - db_size_before:+,} bytes")
print("→ The DB file barely changes! All writes go to the WAL first.")
print("→ This is why WAL mode is faster for writes: sequential appends instead of random page updates.")

In [ ]:
# Checkpoint: flush WAL back into the main DB file

print("Before checkpoint:")
wal_size_before = os.path.getsize(wal_path)
db_size_before = os.path.getsize(DB_PATH)
print(f"  DB:  {db_size_before:>10,} bytes")
print(f"  WAL: {wal_size_before:>10,} bytes")

# PRAGMA wal_checkpoint forces SQLite to merge WAL back into the DB
result = cur.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone()
# Returns (busy, log_pages, checkpointed_pages)
print(f"\nCheckpoint result: busy={result[0]}, log_pages={result[1]}, checkpointed={result[2]}")

print("\nAfter checkpoint:")
db_size_after = os.path.getsize(DB_PATH)
wal_size_after = os.path.getsize(wal_path) if os.path.exists(wal_path) else 0
print(f"  DB:  {db_size_after:>10,} bytes  (grew by {db_size_after - db_size_before:+,})")
print(f"  WAL: {wal_size_after:>10,} bytes  (truncated!)")

print("\n→ Checkpoint merged all WAL pages back into the B-tree pages in the main file.")
print("→ The WAL is now empty, ready for the next batch of writes.")

In [ ]:
# Performance comparison: WAL mode vs rollback journal mode

def bench_journal_mode(mode, n_rows=10000):
    """Insert n_rows and measure time under a given journal mode."""
    db_path = f"{OUTPUT_DIR}/bench_{mode}.db"
    if os.path.exists(db_path):
        os.remove(db_path)
    for f in glob.glob(f"{db_path}*"):
        os.remove(f)
    
    db = sqlite3.connect(db_path)
    c = db.cursor()
    c.execute(f"PRAGMA journal_mode={mode}")
    c.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, email TEXT, phone TEXT, val REAL)")
    c.execute("CREATE UNIQUE INDEX idx_e ON t(email)")
    c.execute("CREATE UNIQUE INDEX idx_p ON t(phone)")
    
    random.seed(42)
    rows = [(i, f"user{i}@test.com", f"+1-{i:010d}", random.random()) for i in range(n_rows)]
    
    # Bulk insert
    start = time.time()
    c.executemany("INSERT INTO t VALUES (?,?,?,?)", rows)
    db.commit()
    bulk_time = time.time() - start
    
    # Single-row inserts (simulates OLTP)
    start = time.time()
    for i in range(n_rows, n_rows + 1000):
        c.execute("INSERT INTO t VALUES (?,?,?,?)",
                  (i, f"user{i}@test.com", f"+1-{i:010d}", random.random()))
        db.commit()  # commit each row — worst case for journaling
    single_time = time.time() - start
    
    db.close()
    # Cleanup
    for f in glob.glob(f"{db_path}*"):
        os.remove(f)
    
    return bulk_time, single_time

print("Journal Mode Performance (10K bulk + 1K single-row commits)")
print(f"{'Mode':<12} {'Bulk Insert':>12} {'1K Singles':>12} {'Singles/sec':>12}")
print("-" * 52)

for mode in ['DELETE', 'WAL']:
    bulk, single = bench_journal_mode(mode)
    print(f"{mode:<12} {bulk:>10.3f}s {single:>10.3f}s {1000/single:>12,.0f}")

print("\n→ WAL mode is dramatically faster for single-row commits.")
print("  DELETE mode (rollback journal) must fsync the journal AND the DB on every commit.")
print("  WAL mode only appends to the log — sequential I/O, one fsync.")

### WAL: The key insight

```
Without WAL (rollback journal):                With WAL:
  ┌──────────┐                                   ┌──────────┐   ┌──────────┐
  │  DB File  │ ← random page rewrites            │  DB File  │   │ WAL File │
  │ (B-tree)  │   on every commit                  │ (B-tree)  │   │ (append) │
  └──────────┘                                   └──────────┘   └────┬─────┘
                                                                      │
  Crash = corrupted page                          Checkpoint ─────────┘
                                                   (periodic merge)
```

**Why WAL matters for B-trees specifically:**
- A single INSERT may modify 3+ B-tree pages (leaf + parent + index pages)
- Without WAL: each modified page must be written to its exact location in the file (random I/O)
- With WAL: all modified pages are appended sequentially to the log (sequential I/O)
- Sequential I/O is 10-100x faster than random I/O on spinning disks, and still faster on SSDs

This is the universal pattern: **every database separates the "what changed" (log) from the "where it lives" (B-tree pages)**.

In [17]:
# Compare actual SQLite operation times as data grows

def benchmark_operations(n_rows):
    db = sqlite3.connect(":memory:")
    c = db.cursor()
    c.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, val INT, grp INT)")
    c.execute("CREATE INDEX idx_val ON t(val)")
    c.execute("CREATE INDEX idx_grp ON t(grp)")
    c.execute("CREATE TABLE g (id INTEGER PRIMARY KEY, name TEXT)")
    
    random.seed(42)
    groups = 100
    c.executemany("INSERT INTO g VALUES (?,?)", 
                  [(i, f"group_{i}") for i in range(groups)])
    
    c.executemany("INSERT INTO t VALUES (?,?,?)",
                  [(i, random.randint(1, n_rows * 10), random.randint(0, groups-1))
                   for i in range(n_rows)])
    db.commit()
    
    results = {}
    
    # Point lookup
    target = n_rows // 2
    start = time.time()
    for _ in range(1000):
        c.execute("SELECT * FROM t WHERE id = ?", (target,)).fetchone()
    results['point_lookup'] = (time.time() - start) / 1000
    
    # Index scan (range)
    lo, hi = n_rows * 3, n_rows * 4
    start = time.time()
    for _ in range(100):
        c.execute("SELECT * FROM t WHERE val BETWEEN ? AND ?", (lo, hi)).fetchall()
    results['range_scan'] = (time.time() - start) / 100
    
    # Full scan
    start = time.time()
    for _ in range(10):
        c.execute("SELECT COUNT(*) FROM t WHERE val > 0").fetchone()
    results['full_scan'] = (time.time() - start) / 10
    
    # Indexed join
    start = time.time()
    for _ in range(100):
        c.execute("SELECT g.name, COUNT(*) FROM t JOIN g ON t.grp = g.id GROUP BY g.id").fetchall()
    results['indexed_join'] = (time.time() - start) / 100
    
    # Insert (single row)
    start = time.time()
    for i in range(1000):
        c.execute("INSERT INTO t VALUES (?,?,?)", 
                  (n_rows + i + 1, random.randint(1, n_rows * 10), random.randint(0, 99)))
    results['insert'] = (time.time() - start) / 1000
    
    db.close()
    return results

sizes = [1000, 5000, 10000, 50000, 100000]
all_results = {}

print("Benchmarking operations across data sizes...")
for n in sizes:
    print(f"  {n:>7,} rows...", end=" ", flush=True)
    all_results[n] = benchmark_operations(n)
    print("done")

print("\nResults (milliseconds):")
ops = ['point_lookup', 'insert', 'range_scan', 'indexed_join', 'full_scan']
header = f"{'Rows':>8} " + " ".join(f"{op:>14}" for op in ops)
print(header)
print("-" * len(header))
for n in sizes:
    vals = " ".join(f"{all_results[n][op]*1000:>14.3f}" for op in ops)
    print(f"{n:>8,} {vals}")

Benchmarking operations across data sizes...
    1,000 rows... done
    5,000 rows... done
   10,000 rows... done
   50,000 rows... done
  100,000 rows... done

Results (milliseconds):
    Rows   point_lookup         insert     range_scan   indexed_join      full_scan
-----------------------------------------------------------------------------------
   1,000          0.003          0.003          0.079          0.134          0.012
   5,000          0.001          0.002          0.246          0.182          0.034
  10,000          0.001          0.002          0.370          0.315          0.061
  50,000          0.001          0.002          2.505          1.440          0.275
 100,000          0.001          0.002          5.046          2.879          0.554


In [ ]:
# Plot scaling behavior
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'point_lookup': 'green', 'insert': 'orange', 
          'range_scan': 'blue', 'indexed_join': 'purple', 'full_scan': 'red'}
labels = {'point_lookup': 'Point Lookup O(log N)', 'insert': 'Insert O(log N)',
          'range_scan': 'Range Scan O(log N + K)', 
          'indexed_join': 'Indexed Join O(M·log N)',
          'full_scan': 'Full Scan O(N)'}

for op in ops:
    times = [all_results[n][op] * 1000 for n in sizes]
    ax.plot(sizes, times, 'o-', color=colors[op], label=labels[op], linewidth=2)

ax.set_xlabel('Number of Rows', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('B-Tree Operation Scaling in SQLite', fontsize=14)
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/operation_scaling.png', dpi=150)
plt.show()

print("\nKey observations:")
print("  • Point lookups barely grow — logarithmic scaling is powerful")
print("  • Full scans grow linearly — touching every page")
print("  • Inserts grow slowly but carry the split overhead")
print("  • Indexed joins scale with outer table × log(inner table)")

---
## Summary: The B-Tree Mental Model

```
                    ┌─────────────────────────────┐
                    │     ROOT (1 page read)       │
                    └──────────┬──────────────────┘
                               │
                    ┌──────────▼──────────────────┐
                    │   INTERNAL (1 page read)     │
                    └──────────┬──────────────────┘
                               │
                    ┌──────────▼──────────────────┐
                    │     LEAF (data lives here)   │
                    └─────────────────────────────┘

Height 3 = 3 page reads = 3 disk I/Os for ANY lookup
regardless of whether you have 1K or 1M rows.
```

### What you should take away:

1. **Reads are cheap**: O(log_B N) — usually 2-4 page reads for millions of rows
2. **Writes are expensive**: same traversal + insert + possible split cascade
3. **Each index multiplies write cost**: INSERT touches N+1 B-trees (table + N indexes)
4. **Index high-cardinality columns**: unique/near-unique columns (email, phone) benefit most from B-trees. Low-cardinality columns (department, status) don't.
5. **WAL decouples writes from B-tree pages**: append to a sequential log first, merge into B-tree later. This is how every serious database works.
6. **Tree height rarely changes**: grows only when root splits — logarithmic growth

### Coming up next: LSM-Trees (RocksDB)

LSM-Trees flip the tradeoff: **writes are cheap, reads pay the cost**.
Instead of maintaining a sorted tree on every write, they buffer writes and sort lazily.
The WAL concept we just saw is a stepping stone toward understanding LSM — an LSM-tree is essentially "what if the WAL *was* the primary data structure?"

In [ ]:
# Cleanup
conn.close()
print("Lab complete! Files generated in _output/:")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f):<30} {size:>10,} bytes")